# Energy pricing calibration — 01 source extraction

Single source of truth for the energy-pricing source register. Every row is
written by this notebook; `data/sources_register.csv` is a generated artifact
and must never be hand-edited.

Run this before `02_energy_pricing_calibration.ipynb`, which fails rather than
citing a source that does not resolve against the register.

Stored documents follow `{source_id}.{ext}` — the infrastructure source ids
already encode country, document and year, so the filename is derivable from
the register and needs no column of its own.

In [ ]:
# Energy pricing calibration — source extraction
#
# Same contract as the TAC calibration: notebook is truth, CSV is output.
# Documents shared with TAC (network statements that carry both a track
# access rate and a separately excluded electric-supply-equipment charge)
# are re-declared here rather than cross-read, so each domain register
# stands alone and neither notebook depends on the other's outputs.

import csv
from pathlib import Path


def _resolve_data_dir() -> Path:
    """Notebook may run from calib/ or from the repo root; resolve either."""
    here = Path.cwd()
    for cand in (
        here / "data",
        here / "backend/models/infrastructure/energy_pricing/calib/data",
    ):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the calib data directory from {here}")


DATA_DIR = _resolve_data_dir()
print(f"data directory: {DATA_DIR}")

# Date the register was last reviewed end to end. Held in one place so a
# review pass cannot leave rows carrying stale or drifted access dates.
REGISTER_REVIEWED = "2026-08-17"

REGISTER_COLUMNS = [
    "source_id",
    "short_id",
    "used",
    "downloaded",
    "title",
    "publisher",
    "pub_year",
    "price_basis_year",
    "currency",
    "kind",
    "url_or_file",
    "date_accessed",
    "reliability_note",
]

register_rows: list[tuple] = []

## Price documents

The commodity price, the traction-network tariffs and the national all-in
tariffs — the three derivation modes of §2–§3 of the generated document.

In [ ]:
# --- Price documents ---
_R = REGISTER_REVIEWED

register_rows += [
    (
        "EUROSTAT-NRG-PC-205-C",
        "eurostat_nrg",
        "Used",
        "x",
        "Electricity price components for non-household consumers (nrg_pc_205_c)",
        "Eurostat",
        2026,
        2025,
        "EUR",
        "official_statistics",
        "https://ec.europa.eu/eurostat/databrowser/view/nrg_pc_205_c/default/table",
        _R,
        "Band IE (20,000-69,999 MWh/a) is the rail-relevant band for a night "
        "train operation and is validated against two independent IM tariffs "
        "(HU and SE match band IE almost exactly, neither matches band IA). "
        "Nine price components per country; the benchmark for every country "
        "without a published traction-energy tariff",
    ),
    (
        "DESNZ-T341",
        "desnz_t341",
        "Used",
        "x",
        "Prices of fuels purchased by non-domestic consumers, Tables 3.4.1 and 3.4.2",
        "Department for Energy Security and Net Zero",
        2026,
        2025,
        "GBP",
        "official_statistics",
        "https://www.gov.uk/government/statistical-data-sets/gas-and-electricity-prices-in-the-non-domestic-sector",
        _R,
        "GB is not in nrg_pc_205_c. The 'Large' band is defined identically to "
        "Eurostat band IE, so the two series are directly comparable. The "
        "excl.-CCL column is the working series (rail traction is CCL-exempt); "
        "the gap to the incl.-CCL column is a survey average across liable and "
        "exempt consumers and must not be read as the statutory CCL rate",
    ),
    (
        "CH-NZV",
        "ch_nzv",
        "Used",
        "x",
        "SR 742.122 Eisenbahn-Netzzugangsverordnung (NZV)",
        "Swiss Confederation (Fedlex)",
        2026,
        2027,
        "CHF",
        "regulation",
        "https://www.fedlex.admin.ch/eli/cc/1999/142/de#a21",
        _R,
        "Art.20a transitional traction-current tariff: 0.13 CHF/kWh base and a "
        "40% reduced rate 22:00-06:00. All-in - Switzerland levies no separate "
        "electricity excise and no separate supply-equipment charge, so this "
        "single price replaces the whole component stack",
    ),
    (
        "AT-SNNB-2026",
        "at_snnb_2026",
        "Used",
        "x",
        "Schienennetz-Nutzungsbedingungen 2026",
        "ÖBB-Infrastruktur AG",
        2024,
        2026,
        "EUR",
        "network_statement",
        "https://infrastruktur.oebb.at/de/geschaeftspartner/schienennetz/snnb",
        _R,
        "Tab.59 p.114 Bahnstromnetz: Hochtarif 52.40 and Niedertarif 43.67 "
        "EUR/MWh, the latter 22:00-06:00. Single-part tariff billed on energy "
        "drawn, so it converts to EUR/kWh without a demand assumption. The 2026 "
        "edition is used deliberately: the 2027 edition supersedes it for TAC "
        "but the electricity table is the one published here",
    ),
    (
        "DE-DBE-NETZ-2026",
        "de_dbe_netz_2026",
        "Used",
        "x",
        "Preisblatt Netznutzung Bahnstromnetz, from 01.01.2026",
        "DB Energie",
        2025,
        2026,
        "EUR",
        "tariff_list",
        "https://www.dbenergie.de/dbenergie-de/Produkte/bahnstrom",
        _R,
        "Two-part tariff (Leistungspreis + Arbeitspreis) on the 16.7 Hz traction "
        "network, so the per-kWh equivalent depends on the RU's own "
        "Benutzungsdauer - derived in §3b and banded, not assumed open. The "
        "statutory levies named in §7 are not quantified in the sheet and are "
        "deliberately not added: they sit in Eurostat's tax columns already",
    ),
    (
        "FR-DRR-A512",
        "fr_drr_a512",
        "Used",
        "x",
        "DRR 2027 Annexe 5.1.2 — tarification de la traction électrique",
        "SNCF Réseau",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.sncf-reseau.com/en/drr/network-statement-national-rail-network-timetable-2027",
        _R,
        "Publishes the RCTE-A loss formula and the loss-rate series, which is "
        "what makes the 2027 network override derivable before SNCF Réseau "
        "publishes the tariff itself in December 2026",
    ),
    (
        "FR-DRR-2024-A52",
        "fr_drr_2024_a52",
        "Used",
        "x",
        "DRR 2024 Annexe 5.2 v12",
        "SNCF Réseau",
        2024,
        2024,
        "EUR",
        "tariff_annex",
        "https://www.sncf-reseau.com/fr/reseau/documents-de-reference-du-reseau",
        _R,
        "The only edition carrying a published RCTE-A value (0.02912 EUR/kWh). "
        "Used to validate the loss formula by back-solving it, not as a working "
        "value - the 2024 figures are crisis-era purchase prices",
    ),
    (
        "FR-DRR-2024-A54",
        "fr_drr_2024_a54",
        "Used",
        "x",
        "DRR 2024 Annexe 5.4 v11",
        "SNCF Réseau",
        2024,
        2024,
        "EUR",
        "tariff_annex",
        "https://www.sncf-reseau.com/fr/reseau/documents-de-reference-du-reseau",
        _R,
        "RCTE-B 0.02003 and RFE 0.18653 EUR/kWh. The RFE figure is what confirms "
        "the back-solved purchase price and therefore the formula",
    ),
    (
        "HR-NS-2027",
        "hr_ns_2027",
        "Used",
        "x",
        "Izvješće o mreži / Network Statement 2027",
        "HŽ Infrastruktura",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://eng.hzinfra.hr/?page_id=284",
        _R,
        "Ch.5.4 item 40: traction energy at a night (NT) and day (VT) rate plus "
        "a renewables levy, and separately the 0.10 EUR/train-km electric-"
        "traction surcharge that the TAC calibration excluded as energy. Excise "
        "is printed as 0.000000, which overrides the CE Delft PPS figure",
    ),
    (
        "HU-NS-2627",
        "hu_ns_2627",
        "Used",
        "x",
        "Network Statement 2026-2027, Annex 5.2-6",
        "VPE / MÁV",
        2026,
        2027,
        "HUF",
        "network_statement",
        "https://vpe.kti.hu/en/network-statement/network-statement-2026-2027/",
        _R,
        "All-in traction price 67.4 HUF/kWh (49.0 energy + 10.2 system + 0.4 "
        "excise + 7.8 funds) and the 110 HUF per electric train-km catenary-use "
        "charge. The all-in price matches Eurostat band IE HU to within 1%, "
        "which is one of the two validations of the band choice",
    ),
    (
        "SE-NS-2027",
        "se_ns_2027",
        "Used",
        "x",
        "Network Statement 2027, §7.3.11",
        "Trafikverket",
        2026,
        2027,
        "SEK",
        "network_statement",
        "https://bransch.trafikverket.se/en/startpage/operations/Operations-railway/Network-Statement/network-statement-2027/",
        _R,
        "Worked example 0.7156 SEK/kWh (0.6075 energy + 0.1081 grid). Energy and "
        "grid together, so no separate supply-equipment charge applies. Sits "
        "just below Eurostat band IE SE - the second validation of the band",
    ),
]
print(f"{len(register_rows)} price documents")

## Tax and VAT documents

The rail-specific electricity excise, which replaces the standard rate
Eurostat reports, and the one confirmed case of non-deductible input VAT.

In [ ]:
# --- Tax and VAT documents ---
register_rows += [
    (
        "CE-DELFT-4K83",
        "ce_delft_4k83",
        "Used",
        "x",
        "Transport taxes and charges in Europe (4.K83) and accompanying database",
        "CE Delft for the European Commission",
        2019,
        2016,
        "EUR",
        "study",
        "https://cedelft.eu/publications/transport-taxes-and-charges-in-europe/",
        _R,
        "Sheet Rail_Energy taxes_level: the per-country rail electricity tax the "
        "report only plots. Values are PPS-adjusted, which moves the working "
        "price by well under 0.5% since the largest entry is 1.4% of a typical "
        "energy price. The only pan-European source on rail excise exemptions",
    ),
    (
        "DE-APS-2027",
        "de_aps_2027",
        "Used",
        "x",
        "Anlagenpreissystem 2027 — Entgelte für Serviceeinrichtungen",
        "DB InfraGO",
        2025,
        2027,
        "EUR",
        "facility_price_list",
        "https://www.dbinfrago.com/web/schienennetz/regelwerke-nutzungsbedingungen/preise",
        _R,
        "States the StromStG rates directly: 20.50 EUR/MWh standard and 11.42 "
        "EUR/MWh reduced under §9(2) on presentation of an Erlaubnisschein - "
        "identical to the 2016 CE Delft figure, so the rail rate has not moved "
        "in a decade. Also prices stabling and pre-heating energy, which belong "
        "to the facility domain and are out of scope here",
    ),
    (
        "APS-STROMSTEUER",
        "aps_stromsteuer",
        "Used",
        "x",
        "EU-Vergleich Besteuerung von Eisenbahn-Fahrstrom",
        "Allianz pro Schiene",
        2016,
        2016,
        "EUR",
        "advocacy_analysis",
        "https://www.allianz-pro-schiene.de/themen/umwelt/energieverbrauch/",
        _R,
        "Nominal rates rather than PPS-adjusted. Used only to corroborate the "
        "Austrian Elektrizitätsabgabe at 15.00 EUR/MWh, which has no rail "
        "carve-out; superseded by CE-DELFT-4K83 wherever the two disagree",
    ),
    (
        "SKAT-DK-VAT",
        "skat_dk_vat",
        "Used",
        "x",
        "Services exempt from VAT",
        "Skattestyrelsen (Danish Tax Agency)",
        2024,
        2024,
        "DKK",
        "tax_authority",
        "https://skat.dk/en-us/businesses/vat/services-exempt-from-vat",
        _R,
        "Danish passenger transport is VAT-exempt, and exemption removes the "
        "right to deduct input VAT - so VAT on electricity is a real cost in "
        "Denmark and nowhere else yet confirmed. This is the single assumption "
        "in the calibration that moves a working price by 15-30%",
    ),
]
print(f"{len(register_rows)} rows after tax and VAT documents")

## Electric supply equipment documents

Network statements that charge separately for use of the catenary and the
traction power-supply installations. The TAC calibration excluded every one of
these as energy — this domain is where they are priced, so each exclusion is
picked up here rather than lost between the two.

In [ ]:
# --- Electric supply equipment documents ---
# Every row corresponds to an "excluded (energy)" line in TAC_CALIBRATION.md.
# Three carry a charge whose rate could not be read (BG's unit is ambiguous,
# ES publishes Mode C without a rate in the consulted text, UK's EAUC is in a
# price list section not yet extracted) — they stay in the register as MISSING
# values rather than being dropped, because the charge demonstrably exists.
register_rows += [
    (
        "BE-NS-2027",
        "be_ns_2027",
        "Used",
        "x",
        "Network Statement 2027 (version 30 June 2026)",
        "Infrabel",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://infrabel.be/en/networkstatement",
        _R,
        "§5.3 direct cost for catenary use, 17.22 EUR/MWh - the only "
        "supply-equipment charge in the calibration published per unit of "
        "energy rather than per train-km, so it adds to the Belgian price "
        "per kWh instead of becoming a separate column",
    ),
    (
        "BG-NRIC-2026",
        "bg_nric_2026",
        "Not used",
        "x",
        "Charges and Prices, Annex 5.3.2 v.06",
        "NRIC (National Railway Infrastructure Company)",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://www.rail-infra.bg/en/353",
        _R,
        "Section I lists a charge for use of power-supply equipment at 17.29 "
        "EUR, without a unit that can be read unambiguously (per train-km, per "
        "MWh and per path are all consistent with the surrounding text). "
        "Recorded as MISSING until the Bulgarian text is checked",
    ),
    (
        "ES-BOE-2024",
        "es_boe_2024",
        "Not used",
        "x",
        "BOE-A-2024-22140 (consolidated) — railway charges",
        "Boletín Oficial del Estado",
        2024,
        2023,
        "EUR",
        "regulation",
        "https://www.boe.es/buscar/act.php?id=BOE-A-2024-22140",
        _R,
        "Modality C covers transformation and distribution of traction "
        "electricity. The consulted consolidation names the modality without a "
        "rate table, so the Spanish supply-equipment charge is MISSING rather "
        "than zero",
    ),
    (
        "FI-NS-2027",
        "fi_ns_2027",
        "Used",
        "x",
        "Verkkoselostus / Network Statement 2027",
        "Väylävirasto (Finnish Transport Infrastructure Agency)",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.doria.fi/bitstream/handle/10024/195216/vj_2026-38eng_978-952-405-425-6.pdf?sequence=1&isAllowed=y",
        _R,
        "Tab.2 §5.3 additional charge for electric supply equipment, 0.0167 "
        "cents per gross-tonne-km. Finland charges everything per gross-tonne-"
        "km, this term included",
    ),
    (
        "FR-DRR-2027-A52",
        "fr_drr_2027_a52",
        "Used",
        "x",
        "DRR 2027 Appendix 5.2 — scale of minimum services",
        "SNCF Réseau",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.sncf-reseau.com/en/drr/network-statement-national-rail-network-timetable-2027",
        _R,
        "RCE 0.291 EUR per electric train-km: an infrastructure charge for use "
        "of the catenary, distinct from the RCTE components that price the "
        "energy itself. The largest supply-equipment charge in the calibration",
    ),
    (
        "GR-OSE-2026",
        "gr_ose_2026",
        "Used",
        "x",
        "Network Statement 2026 (EN final)",
        "OSE",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://ose.gr/wp-content/uploads/2026/02/OSE_2026-ENG_Final.pdf",
        _R,
        "Ch.6 electrification wear c_wte 0.00210 EUR per tonne-km, subject to "
        "the same inflation multiplier (1.1975) and phased recovery factor "
        "(0.60) as the track terms - so the effective rate is derived, not read",
    ),
    (
        "IE-NS-2027",
        "ie_ns_2027",
        "Used",
        "x",
        "Network Statement 2027",
        "Iarnród Éireann",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.irishrail.ie/en-ie/about-us/iarnrod-eireann-network-statement",
        _R,
        "The traction-power charge applies to the DART network only. The Irish "
        "intercity network a night train would use is unelectrified, so the "
        "charge is positively not levied rather than missing",
    ),
    (
        "IT-LISTINO",
        "it_listino",
        "Used",
        "x",
        "Listino Tariffario Pacchetto Minimo di Accesso (PMdA)",
        "RFI",
        2026,
        2027,
        "EUR",
        "tariff_list",
        "https://www.rfi.it/en/railway-infrastructure-access-/Network-statement.html",
        _R,
        "Component TA3, contact-line use: 0.241 EUR/train-km on the "
        "conventional network and 0.482 on high speed. Which applies depends on "
        "the line a leg runs on, which the model does not resolve yet - hence "
        "the conventional rate with the pair as the band",
    ),
    (
        "LT-LTG-2627",
        "lt_ltg_2627",
        "Used",
        "x",
        "Network Statement 2026-2027 and annexes v1",
        "LTG Infra",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://ltginfra.lt/en/railway-infrastructure/map/network-statements/",
        _R,
        "§5.3.2 contact-network fee 0.1709 EUR/train-km, alongside a track "
        "charge levied purely per gross-tonne-km",
    ),
    (
        "LU-NS-2027",
        "lu_ns_2027",
        "Used",
        "x",
        "Document de référence du réseau 2027 (EN v1.0)",
        "ACF / CFL",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://acf.gouvernement.lu/en/sillon/Document-de-reference-du-reseau.html",
        _R,
        "§5.3.2 electric supply c_E 0.2583 EUR/train-km, a flat term with none "
        "of the length and category factors the Luxembourg track charge carries",
    ),
    (
        "LV-NS-2027",
        "lv_ns_2027",
        "Used",
        "x",
        "Network Statement 2027",
        "LDz / LatRailNet",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://www.ldz.lv/en/network-statement-2027",
        _R,
        "§5.2 electric-traction supply-equipment charge 0.15 EUR/train-km for "
        "international passenger services within the EEA",
    ),
    (
        "PL-PLK-A91",
        "pl_plk_a91",
        "Used",
        "x",
        "Network Statement 2026/2027 Annex 9.1 (SMK)",
        "PKP Polskie Linie Kolejowe",
        2026,
        2027,
        "PLN",
        "network_statement",
        "https://en.plk-sa.pl/for-customers-and-partners/the-rules-for-allocating-train-paths/network-statement-2026/2027",
        _R,
        "Electric-traction component 0.29 PLN per km, flat and outside the WM "
        "(mass) and WK (line category) factor chain that scales the base rate",
    ),
    (
        "RO-CFR-A25",
        "ro_cfr_a25",
        "Used",
        "x",
        "Network Statement Annex 25.a / 26.a",
        "CFR SA",
        2025,
        2024,
        "RON",
        "network_statement",
        "https://cfr.ro/download-drr-2026-network-statement/",
        _R,
        "Electrification component Ttse 0.676 lei/train-km. Rates valid from "
        "1 Mar 2024, so this is the longest escalation path in the domain "
        "apart from the statutory taxes",
    ),
    (
        "SK-ZSR-A52B",
        "sk_zsr_a52b",
        "Used",
        "x",
        "Network Statement 2027 Annex 5.2.B (Measure 2/2018)",
        "ŽSR",
        2026,
        2019,
        "EUR",
        "network_statement",
        "https://www.zsr.sk/en/railway-undertaking/infrastructure/network-statement/network-statement-2027/",
        _R,
        "Component U4, electric supply equipment, 0.228 EUR per 1,000 "
        "gross-tonne-km. Frozen with the rest of Measure 2/2018 since 2019, "
        "which is why it carries the same 0%/yr escalation deviation as the "
        "Slovak track charges",
    ),
    (
        "UK-NR-CP7",
        "uk_nr_cp7",
        "Not used",
        "x",
        "CP7 Track Usage Price List",
        "Network Rail",
        2024,
        2024,
        "GBP",
        "tariff_list",
        "https://www.networkrail.co.uk/industry-and-commercial/information-for-operators/network-statement/",
        _R,
        "Carries the Electrification Asset Usage Charge, which the TAC "
        "calibration excluded as energy. The rate sits in a price-list sheet "
        "not yet extracted, so the GB supply-equipment charge is MISSING",
    ),
]
print(f"{len(register_rows)} rows after supply equipment documents")

## Documents still to check, and the conversion sources

Seven network statements whose electric-supply-equipment position is unknown:
the TAC extraction recorded no energy exclusion for them, which is weak
evidence of absence rather than a documented zero. They are registered as
`Not used` so the gap is visible and addressable, not silently priced at zero.

In [ ]:
# --- Documents still to check ---
# Registered but not yet cited by a value. A country here carries a MISSING
# supply-equipment charge, which the seed export writes as NULL and the cost
# model therefore prices at zero — an understatement, and the reason these
# rows exist rather than being left out.
register_rows += [
    (
        "CZ-NS-2027",
        "cz_ns_2027",
        "Not used",
        "x",
        "Network Statement 2027 (EN web version)",
        "Správa železnic",
        2026,
        2027,
        "CZK",
        "network_statement",
        "https://www.spravazeleznic.cz/web/en/network-statement-2027",
        _R,
        "Charging annex to check for a separate traction-current or "
        "supply-equipment term",
    ),
    (
        "DK-NS-2027",
        "dk_ns_2027",
        "Not used",
        "-",
        "Network Statement 2027",
        "Banedanmark",
        2026,
        2027,
        "DKK",
        "network_statement",
        "https://www.bane.dk/en/Railway/Network-Statement",
        _R,
        "Danish charges are set by executive order rather than in the network "
        "statement; check the order for an electrification term",
    ),
    (
        "EE-TTJA-2026",
        "ee_ttja_2026",
        "Not used",
        "x",
        "Raudteeinfrastruktuuri kasutustasu määrad (published rate table)",
        "Tarbijakaitse ja Tehnilise Järelevalve Amet (TTJA)",
        2026,
        2026,
        "EUR",
        "tariff_decision",
        "https://ttja.ee/ariklient/raudtee/kasutustasu-maarad",
        _R,
        "Rate table to check; the Estonian main line a night train would use is "
        "electrified only in part",
    ),
    (
        "NL-NS-2027",
        "nl_ns_2027",
        "Not used",
        "x",
        "Network Statement 2027 (version 1.1, 6 May 2026)",
        "ProRail",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://www.prorail.nl/samenwerken/vervoerders/network-statement",
        _R,
        "Check whether the train path service price includes catenary use or "
        "whether ProRail charges it separately",
    ),
    (
        "NO-NS-2027",
        "no_ns_2027",
        "Not used",
        "x",
        "Network Statement 2027 (EN v1.1)",
        "Bane NOR",
        2026,
        2026,
        "NOK",
        "network_statement",
        "https://oppslagsverk.banenor.no/en/network-statement/",
        _R,
        "§5.3.3 to check for an electrification term alongside the train path charge",
    ),
    (
        "PT-IP-2027",
        "pt_ip_2027",
        "Not used",
        "x",
        "1st Addenda to Network Statement 2027",
        "Infraestruturas de Portugal",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://servicos.infraestruturasdeportugal.pt/sites/default/files/1st%20Addenda%20Network%20Statement%202027_0.pdf",
        _R,
        "§5.3 distinguishes electric from diesel traction, so a separate "
        "supply-equipment element is plausible and worth checking",
    ),
    (
        "SI-NS-2027",
        "si_ns_2027",
        "Not used",
        "x",
        "Program omrežja / Network Statement 2027",
        "SŽ-Infrastruktura",
        2026,
        2027,
        "EUR",
        "network_statement",
        "https://infrastruktura.sz.si/en/partners/access-to-infrastructure-for-rus/network-statement/",
        _R,
        "§5.3 factor chain to check for an electric-traction factor or term",
    ),
]

# --- Conversion and method sources ---
# The documents above give a price in a currency at a price basis. Two
# conversions stand between that and a number the cost model can use, and each
# needs its own provenance.
register_rows += [
    (
        "ECB-FX",
        "ecb_fx",
        "Used",
        "-",
        "ECB euro foreign exchange reference rates",
        "European Central Bank",
        2026,
        2026,
        "EUR",
        "fx_reference",
        "https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/index.en.html",
        _R,
        "Reference rates behind FX_TO_EUR, pinned at the same snapshot date as "
        "the TAC calibration so the two infrastructure domains reach EUR on "
        "identical terms. Four of the calibrated prices publish in a non-euro "
        "currency",
    ),
    (
        "ECB-PROJECTIONS",
        "ecb_projections",
        "Used",
        "-",
        "Eurosystem staff macroeconomic projections for the euro area",
        "European Central Bank",
        2025,
        None,
        "EUR",
        "macro_projection",
        "https://www.ecb.europa.eu/press/projections/html/index.en.html",
        _R,
        "The HICP path the 2%/yr energy escalation is anchored on. Electricity "
        "differs from track access here: European wholesale power forwards are "
        "flat to falling in real terms, so nominal HICP is the defensible "
        "carry rather than the 3%/yr real-terms rise track charges show",
    ),
]
print(f"{len(register_rows)} rows in total")

## Write and validate

In [ ]:
# --- Write and validate ---


def write_data(name: str, columns: list[str], rows: list[dict]) -> None:
    """STDLIB-ONLY writer, shared by both calib notebooks."""
    path = DATA_DIR / name
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {c: ("" if row.get(c) is None else row[c]) for c in columns}
            )
    print(f"  {name}: {len(rows)} rows")


register_dicts = [dict(zip(REGISTER_COLUMNS, r)) for r in register_rows]

ids = [r["source_id"] for r in register_dicts]
assert len(ids) == len(set(ids)), (
    f"duplicate source_id: {sorted({i for i in ids if ids.count(i) > 1})}"
)

# Every row must carry a locator a reader can follow — either a public URL or
# an explicit MISSING. Silence is the failure mode this guards against.
_blank = [r["source_id"] for r in register_dicts if not r["url_or_file"]]
assert not _blank, f"rows with no url_or_file: {_blank}"

_gathered = sum(1 for r in register_dicts if r["downloaded"] == "x")

write_data("sources_register.csv", REGISTER_COLUMNS, register_dicts)
print(
    f"register: {len(register_dicts)} sources, "
    f"{sum(1 for r in register_dicts if r['used'] == 'Used')} in use, "
    f"{_gathered} documents on disk"
)